[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/255ribeiro/python_build123d_basics/blob/master/docs/tuto_colab_build/build123d_basic_modo_builder.ipynb)

# build123d: modo Builder
## Exemplos semelhantes ao notebook base

Neste notebook, usamos o estilo Builder do build123d, em que a geometria é montada em etapas, como em um histórico de operações CAD.

O objetivo é mostrar como a mesma lógica de construção pode ser expressa de forma sequencial e clara.

## Instalação

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    import subprocess
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "cadquery-simpleviewer[build123d,interactive]"],
        check=True,
    )
    # build123d pulls in a newer ipython than Colab's kernel bootstrap
    # tolerates. Put Colab's version back on disk — do NOT restart the
    # runtime, the current kernel already has the working ipython loaded.
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "ipython==7.34.0", "--no-deps"],
        check=True,
    )

else:
    print("Not running in Colab, skipping package installation.")


## Importação

In [17]:
import build123d as b3d
from cadquery_simpleviewer import show

---

## 1) Primeiro exemplo em modo Builder

A estrutura básica é: `with b3d.BuildPart() as parte:` e, dentro do bloco, chamamos as primitivas (`b3d.Box`, `b3d.Cylinder` etc.). O resultado final fica em `parte.part`.

In [22]:
with b3d.BuildPart() as parte_caixa_simples:
    # Primitiva única para demonstrar a estrutura mínima do modo Builder
    b3d.Box(80, 60, 20)

modelo1 = parte_caixa_simples.part
show(modelo1)

---

## 2) Base com pilar

Neste exemplo, criamos uma base e depois um pilar acima dela. Esse padrão é útil para entender a lógica de empilhamento de volumes.

In [23]:
with b3d.BuildPart() as parte_base_pilar:
    b3d.Box(120, 80, 12)
    # Pilar apoiado sobre a base (centro em Z=37 para altura 50)
    with b3d.Locations((0, 0, 37)):
        b3d.Cylinder(radius=15, height=50)
    # Esfera posicionada no topo do pilar
    with b3d.Locations((0, 0, 74)):
        b3d.Sphere(radius=12)

modelo2 = parte_base_pilar.part
show(modelo2)

---

## 3) Bloco com recorte

A subtração de volumes pode ser feita com `mode=b3d.Mode.SUBTRACT` nas primitivas. Esse é o passo para criar furos, vazios e cortes.

In [24]:
with b3d.BuildPart() as parte_com_recortes:
    b3d.Box(100, 60, 25)
    # Dois cilindros removem material da caixa
    with b3d.Locations((-20, 0, 0), (20, 0, 0)):
        b3d.Cylinder(radius=10, height=40, mode=b3d.Mode.SUBTRACT)

modelo3 = parte_com_recortes.part
show(modelo3)

---

## 4) Estrutura arquitetônica simples

Agora montamos uma base, quatro pilares e uma laje. O raciocínio é muito parecido com um histórico de operações de projeto.

In [25]:
with b3d.BuildPart() as parte_estrutura_arq:
    b3d.Box(200, 160, 12)

    # Quatro pilares distribuídos nos cantos internos da base
    with b3d.Locations((-70, -60, 46), (70, -60, 46), (-70, 60, 46), (70, 60, 46)):
        b3d.Box(18, 18, 80)

    # Laje superior apoiada sobre os pilares
    with b3d.Locations((0, 0, 97)):
        b3d.Box(180, 140, 10)

modelo4 = parte_estrutura_arq.part
show(modelo4)

---

## Conclusão

O modo Builder é útil quando queremos construir objetos em etapas, com clareza e ordem. Ele é especialmente prático para projetos que envolvem base, pilares, cortes e sobreposições.

---

## Resumo rapido: quando usar Builder vs modo algebrico

- Use **modo Builder** quando o objetivo for ensinar ou documentar um processo passo a passo (base -> pilares -> laje -> recortes).
- Use **modo Builder** quando voce quiser organizar melhor operacoes com contexto, como `Locations`, `Plane` e `mode=SUBTRACT`.
- Use **modo algebrico** quando voce precisar de prototipagem rapida com expressoes diretas (`a + b - c`).
- Use **modo algebrico** quando o modelo for simples e puder ser entendido em poucas linhas.
- Em projetos maiores, uma estrategia comum e combinar os dois: Builder para montar componentes e algebra para combinar conjuntos finais.